# Stage 1 — 20x soma/process segmentation

A start-to-finish, VS Code-friendly workflow: resolve only approved motion-corrected acquisitions, build small local reference maps with one bounded server read, curate somas before detecting processes, curate missing ridge skeletons, group mixed soma/process ROIs, and optionally extract traces.

## 1. Setup — approved acquisitions from a local database snapshot

In [ ]:
%load_ext autoreload
%autoreload 2

import json, os, sys, time
from pathlib import Path
import numpy as np

sys.path.insert(0, '..')
from analysis.session.devshim import LocalGroup
from analysis.seg_20x import Segmentation20xState, build_reference_images, resolve_approved_group
from analysis.seg_20x.gui import launch

GROUP_ID = 212
MANIPULATION = 'ketxyl'
REFERENCE_ACQ_ID = None  # None -> first approved acquisition
REFERENCE_N_FRAMES = 120
PORTABLE_MASK_BUNDLE = None  # None -> newest published server bundle; or set an explicit Path

SCRATCH = Path(os.environ.get('ODYN_SCRATCH_ROOT', str(Path.home() / 'odyn_scratch')))
SCRATCH.mkdir(parents=True, exist_ok=True)
MAIN = Path(os.environ.get('ODYN_IMAGING_ROOT', '/Volumes/MossLab/ImagingData'))
LIVE_DB = MAIN / '.odyn' / 'odyn.db'
SNAPSHOT = SCRATCH / 'odyn_snapshot.db'
# Approval is an explicit input to this workflow, so always take a fresh,
# lock-free raw copy rather than accepting a snapshot from before approval.
group = LocalGroup(LIVE_DB, MAIN, snapshot_to=SNAPSHOT, refresh=True)
print('database snapshot:', json.dumps(group.snapshot, indent=2))
inputs = resolve_approved_group(group, GROUP_ID)
print(json.dumps(inputs.summary(), indent=2))

The resolver has no directory fallback: only `mcor_files.approved == 1` is eligible. It does not require trial rows, so acquisition-only groups such as 198 resolve through the same path.

## 2. Build or load local reference images

In [ ]:
reference_index = 0 if REFERENCE_ACQ_ID is None else inputs.acq_ids.index(int(REFERENCE_ACQ_ID))
reference_acq_id = inputs.acq_ids[reference_index]
reference_path = inputs.paths[reference_index]
reference_dir = SCRATCH / 'seg_20x' / f'group{GROUP_ID}'
reference_dir.mkdir(parents=True, exist_ok=True)
reference_npz = reference_dir / f'reference_acq{reference_acq_id}_{REFERENCE_N_FRAMES}frames.npz'
reference_json = reference_npz.with_suffix('.json')

if reference_npz.exists() and reference_json.exists():
    with np.load(reference_npz) as cached:
        reference = {name: cached[name] for name in cached.files}
    reference_meta = json.loads(reference_json.read_text())
    print('reference cache HIT:', reference_npz)
else:
    t0 = time.time()
    reference, reference_meta = build_reference_images(reference_path, n_frames=REFERENCE_N_FRAMES)
    np.savez_compressed(reference_npz, **reference)
    reference_json.write_text(json.dumps(reference_meta, indent=2))
    print(f'reference cache MISS: one approved movie read in {time.time()-t0:.1f}s')
print(json.dumps(reference_meta, indent=2))

Reference maps and all tuning are local. The GUI never rereads a TIFF. Change `REFERENCE_ACQ_ID` explicitly if the selected acquisition is atypical; the acquisition and frame interval are recorded beside the cache.

## 3. Ordered segmentation and curation GUI

In [ ]:
ROUND = SCRATCH / 'seg_20x' / f'group{GROUP_ID}' / 'curated_20x_rois.h5'
LEGACY = ROUND.with_suffix('.npz')   # rounds saved before the single-bundle save
if ROUND.exists():
    state = Segmentation20xState.load(ROUND)
    print(f'Resuming saved round in phase {state.phase!r}')
elif LEGACY.exists() and LEGACY.with_suffix('.json').exists():
    state = Segmentation20xState.load(LEGACY)
    print(f'Resuming legacy round in phase {state.phase!r}; saving will write {ROUND.name}')
else:
    state = Segmentation20xState(
        reference['structural'], reference.get('correlation'),
        soma_params={'dog_threshold': 0.12},
        process_params={'global_ridge_pctl': 70.0, 'adaptive_block_px': 11},
    )
gui = launch(save_path=ROUND, state=state)


The five phases are enforced: **tune somas → curate somas → tune processes → curate processes → group ROIs**. Cyan is soma, magenta is process, and yellow is the current group selection or ridge being drawn. Save writes one portable HDF5 bundle holding the curated masks, the detector output they were curated from, the reference images, the parameters, the edits, the ROI table, and the groups.


## 4. Inspect the saved round

In [ ]:
server_output = inputs.paths[0].parents[1] / 'python'
published = sorted(server_output.glob(f'group{GROUP_ID}_*_20x_masks_processed_*.h5'))
if PORTABLE_MASK_BUNDLE is not None:
    bundle_path = Path(PORTABLE_MASK_BUNDLE)
elif published:
    bundle_path = published[-1]
else:
    bundle_path = ROUND
if not bundle_path.exists():
    raise FileNotFoundError('Save from the GUI or set PORTABLE_MASK_BUNDLE to a copied .h5 file')

import h5py
with h5py.File(bundle_path, 'r') as saved:
    soma_labels = saved['masks/soma'][:]
    process_labels = saved['masks/process'][:]
    reference = {name: saved['images'][name][:] for name in saved['images']}
    round_config = json.loads(saved.attrs['config_json'])
print('portable mask bundle:', bundle_path)
print(json.dumps(round_config['summary'], indent=2))


## 5. Finalize masks and traces in the standard 10x session format

In [ ]:
RUN_EXTRACTION = False  # set True only after approving the saved masks
from analysis.session.resolve import resolve_group
from analysis.session.finalize import finalize_session, verify, mask_hash

# Reopen the published bundle here so extraction cannot accidentally use
# masks left in memory from an earlier GUI or inspection run.
if not bundle_path.exists():
    raise FileNotFoundError(f'Portable mask bundle not found: {bundle_path}')
import h5py
with h5py.File(bundle_path, 'r') as saved:
    soma_labels = saved['masks/soma'][:]
    process_labels = saved['masks/process'][:]
    reference = {'structural': saved['images/structural'][:]}
    round_config = json.loads(saved.attrs['config_json'])
print('finalizing from portable mask bundle:', bundle_path)

# Give every soma and process a contiguous final ROI id. Keep the mapping
# (including roi_group_id) in parameters_json inside the HDF5 round.
combined = np.zeros_like(soma_labels, dtype=np.int32)
soma_mask = np.zeros_like(combined)
process_mask = np.zeros_like(combined)
roi_manifest = []
next_roi = 1
for roi_type, source_labels, typed_mask in (
    ('soma', soma_labels, soma_mask), ('process', process_labels, process_mask)
):
    for source_id in np.unique(source_labels[source_labels > 0]):
        pixels = source_labels == source_id
        combined[pixels] = next_roi
        typed_mask[pixels] = next_roi
        roi_manifest.append({
            'roi_id': next_roi, 'roi_type': roi_type, 'source_roi_id': int(source_id),
            'roi_group_id': round_config.get('groups', {}).get(f'{roi_type}:{int(source_id)}'),
        })
        next_roi += 1

session = resolve_group(group, group_id=GROUP_ID, manipulation=MANIPULATION, approved_only=True)
existing = verify(session.output_dir)
already_current = existing['status'] == 'ok' and existing['mask_hash'] == mask_hash(combined)
print('existing:', existing['status'], '(current mask)' if already_current else '')

if RUN_EXTRACTION and not already_current:
    result = finalize_session(
        session, combined,
        per_group_masks={'soma': soma_mask, 'process': process_mask},
        images=reference['structural'],
        segmentation_params={
            'workflow': '20x_soma_process',
            'soma': round_config['soma_params'],
            'process': round_config['process_params'],
            'roi_manifest': roi_manifest,
        },
        curation=round_config['curation'],
        neuropil=False, full_acquisition=True, checkpoint_every=16,
        scratch_dir=SCRATCH, detrend=True,
    )
    print(json.dumps({k: result[k] for k in ('path','mask_png','mask_hash','n_rois')}, indent=2))
    print('verify:', verify(session.output_dir)['status'])
elif RUN_EXTRACTION:
    print('Skipping extraction: the standard HDF5 round already matches this mask.')

The processed HDF5 contains the final masks and traces; 20x finalization does not write a `.mat` sidecar. MATLAB reverses HDF5 array dimensions, so restore the Python axis order when reading:

```matlab
file = "group..._processed_YYYYMMDD.h5";
labels = h5read(file, "/masks/labels").';
somas = h5read(file, "/masks/soma").';
processes = h5read(file, "/masks/process").';
roi = permute(h5read(file, "/traces/roi"), [3 2 1]); % ROI × trial × frame
neuropil = permute(h5read(file, "/traces/neuropil"), [3 2 1]);
time_s = h5read(file, "/traces/time_s");
```


## 6. Group ROIs by proximity and correlation

Grouping uses smoothed ΔF/F₀ from the 4-second odor period and 4 seconds afterward, without odor averaging. Candidate pairs must be spatially close and pass the best correlation at −1, 0, or +1 frame lag; each group may contain at most one soma.


In [ ]:
import matplotlib.pyplot as plt

from analysis.session.store import read_session
from analysis.seg_20x.grouping import (
    GROUPING_DEFAULTS, group_rois, proximity_correlation_profile, traces_from_round,
)

GROUPING = {'max_gap_um': 8.0, 'min_correlation': 0.40, 'max_lag_frames': 1}

round_h5 = sorted(session.output_dir.glob(f'group{GROUP_ID}_*_processed_*.h5'))[-1]
parameters = read_session(round_h5)['attrs']['parameters_json']
roi_manifest = parameters['segmentation']['roi_manifest']
traces, trace_source = traces_from_round(round_h5, roi_manifest)
print(f'{len(traces)} ROI traces from {round_h5.name} ({trace_source})')

# The control comes first: correlation that dies within a micron of the PSF
# means the links below are adjacency, not connectivity.
profile = proximity_correlation_profile(
    soma_labels, process_labels, traces, um_per_px=inputs.um_per_px, params=GROUPING)
print(profile.pivot(index='gap_bin_um', columns='pair_type', values='correlation').round(3))

groups, links = group_rois(
    soma_labels, process_labels, traces, um_per_px=inputs.um_per_px, params=GROUPING)
soma_count = {}
for (kind, _), gid in groups.items():
    soma_count[gid] = soma_count.get(gid, 0) + (kind == 'soma')
print(f'{len(soma_count)} groups over {len(groups)} ROIs '
      f'({len(traces) - len(groups)} left ungrouped); '
      f'{sum(n == 0 for n in soma_count.values())} hold no soma')
assert max(soma_count.values(), default=0) <= 1
print(links.status.value_counts().to_string())
print(links[links.linked].head(20).to_string(index=False))


# Compare individual ROIs with the final group assignments.
roi_keys = [
    *(('soma', int(i)) for i in np.unique(soma_labels[soma_labels > 0])),
    *(('process', int(i)) for i in np.unique(process_labels[process_labels > 0])),
]

def pixels_for(key):
    kind, roi_id = key
    labels = soma_labels if kind == 'soma' else process_labels
    return labels == roi_id

def color_overlay(assignments, *, unassigned_gray=False, alpha=0.85):
    rgba = np.zeros((*soma_labels.shape, 4), dtype=float)
    values = sorted(set(assignments.values()), key=repr)
    colors = {value: plt.cm.turbo(i / max(len(values) - 1, 1))
              for i, value in enumerate(values)}
    for key in roi_keys:
        value = assignments.get(key)
        color = (0.7, 0.7, 0.7, 1.0) if value is None else colors[value]
        rgba[pixels_for(key)] = (*color[:3], alpha)
    return rgba

individual = {key: index for index, key in enumerate(roi_keys)}
ungrouped_rgba = color_overlay(individual)
grouped_rgba = color_overlay(groups, unassigned_gray=True)

structural = reference['structural']
lo, hi = np.percentile(structural[np.isfinite(structural)], (1, 99.5))
fig, axes = plt.subplots(1, 2, figsize=(16, 7), constrained_layout=True)
for ax, overlay, title in (
    (axes[0], ungrouped_rgba, f'Ungrouped: {len(roi_keys)} ROIs'),
    (axes[1], grouped_rgba,
     f'Grouped: {len(set(groups.values()))} groups; '
     f'{len(roi_keys) - len(groups)} ungrouped (gray)'),
):
    ax.imshow(structural, cmap='gray', vmin=lo, vmax=hi)
    ax.imshow(overlay)
    ax.set(title=title, xticks=[], yticks=[])

group_figure = round_h5.with_name(round_h5.stem + '_groups.png')
fig.savefig(group_figure, dpi=160, bbox_inches='tight')
print(f'grouping figure: {group_figure}')
